# Day 3: Sensitive proposals and learned motion assignment

This notebook changes the Day 2 pipeline in four controlled ways:

1. generate a permissive set of multiscale 3D center proposals;
2. refine each peak to a local weighted centroid;
3. estimate whole-frame tissue translation;
4. learn a robust displacement prior from annotated edges and use it in sparse global assignment.

The aim is to improve sensitivity without allowing the final node count to grow unchecked. Model selection uses training movies only. Test images are used once, after the configuration is frozen.


## Rationale and prior work

Sparse annotation changes the learning problem: an unlabelled location is unknown, not necessarily background. The organizer's [metric specification](https://github.com/royerlab/kaggle-cell-tracking-competition/blob/main/metrics.md) therefore motivates high-recall proposals followed by explicit node-count calibration.

[Malin-Mayor et al.](https://www.nature.com/articles/s41587-022-01427-7) combine center evidence, backward motion prediction and graph optimization for whole-embryo lineage reconstruction from sparse annotations. [Hayashida et al.](https://openaccess.thecvf.com/content_CVPR_2020/html/Hayashida_MPM_Joint_Representation_of_Motion_and_Position_Map_for_Cell_CVPR_2020_paper.html) likewise show the value of coupling cell position and motion. The efficient candidate-neighbour design is also consistent with [CELLECT](https://www.nature.com/articles/s41592-025-02886-x), which performs association in a sparse local candidate set.

This first implementation uses a robust motion distribution rather than a large neural network. It is fast, interpretable and provides a direct test of whether motion-aware candidate scoring improves the official edge objective.


## 1. Imports and configuration

The notebook uses only packages available in the Kaggle image. No internet installation is required.


In [ ]:
from pathlib import Path
from itertools import product
from collections import defaultdict
import json, math, time, warnings

import blosc2
import zstandard as zstd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, gaussian_laplace, maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')
np.random.seed(42)
print('numpy', np.__version__, '| blosc2', blosc2.__version__, '| zstandard', zstd.__version__)


In [ ]:
COMPETITION_SLUG = 'biohub-cell-tracking-during-development'
SPACING_UM = np.array([1.625, 0.40625, 0.40625], dtype=np.float32)
MATCH_RADIUS_UM = 7.0

CONFIG = {
    # Detection is permissive here; node-count calibration happens afterward.
    'log_scales_um': [0.9, 1.3, 1.8],
    'background_sigma_um': 4.5,
    'proposal_quantile': 0.990,
    'nms_um': 1.8,
    'refine_radius_um': 1.6,
    'max_proposals_per_frame': 2400,
    # The final count is calibrated against the organizer-provided estimate.
    'count_multipliers': [0.95, 1.00, 1.05],
    # A small candidate graph keeps assignment fast.
    'candidate_k': 6,
    'raw_gate_um': 11.0,
    'residual_gate_um': 8.0,
    'score_weight': 0.12,
}

VALIDATION_MOVIES_PER_EMBRYO = 1
print(json.dumps(CONFIG, indent=2))


## 2. Zarr v3 and GEFF readers

These readers inspect each array's declared codec chain. They support the raw Zstandard chunks used by the annotation graphs as well as Blosc-compressed image chunks.


In [ ]:
def _dtype_from_metadata(meta):
    dtype = np.dtype(meta['data_type'])
    byte_codec = next((c for c in meta.get('codecs', []) if c.get('name') == 'bytes'), None)
    if byte_codec and dtype.itemsize > 1:
        endian = byte_codec.get('configuration', {}).get('endian', 'little')
        dtype = dtype.newbyteorder('<' if endian == 'little' else '>')
    return dtype

def decode_zarr_chunk(raw, meta, expected_nbytes=None):
    """Reverse the compression codecs declared in Zarr v3 metadata."""
    decoded = raw
    for codec in reversed(meta.get('codecs', [])):
        name = codec.get('name')
        if name == 'blosc':
            decoded = blosc2.decompress(decoded)
        elif name == 'zstd':
            decoded = zstd.ZstdDecompressor().decompress(decoded, max_output_size=expected_nbytes or 0)
        elif name == 'bytes':
            pass  # dtype and byte order are applied by np.frombuffer below
        elif name in {'crc32c'}:
            decoded = decoded[:-4]  # checksum bytes follow the encoded payload
        else:
            raise NotImplementedError(f'Unsupported Zarr codec: {name}')
    return decoded

def read_zarr_v3_array(array_path):
    array_path = Path(array_path)
    meta = json.loads((array_path / 'zarr.json').read_text())
    shape = tuple(meta['shape'])
    chunks = tuple(meta['chunk_grid']['configuration']['chunk_shape'])
    dtype = _dtype_from_metadata(meta)
    fill = meta.get('fill_value', 0)
    out = np.full(shape, 0 if fill is None else fill, dtype=dtype)
    grid = tuple(math.ceil(s / c) for s, c in zip(shape, chunks))
    for index in product(*(range(n) for n in grid)):
        path = array_path / 'c'
        for i in index:
            path /= str(i)
        if not path.exists():
            continue
        target = tuple(slice(i*c, min((i+1)*c, s)) for i, c, s in zip(index, chunks, shape))
        actual_shape = tuple(sl.stop-sl.start for sl in target)
        expected_nbytes = math.prod(chunks) * dtype.itemsize
        flat = np.frombuffer(decode_zarr_chunk(path.read_bytes(), meta, expected_nbytes), dtype=dtype)
        if flat.size == math.prod(chunks):
            chunk = flat.reshape(chunks)[tuple(slice(0,n) for n in actual_shape)]
        elif flat.size == math.prod(actual_shape):
            chunk = flat.reshape(actual_shape)
        else:
            raise ValueError(f'Unexpected decoded chunk size at {path}: {flat.size}')
        out[target] = chunk
    return out

class TimeChunkedZarr:
    def __init__(self, path):
        self.path = Path(path) / '0'
        self.metadata = json.loads((self.path / 'zarr.json').read_text())
        self.shape = tuple(self.metadata['shape'])
        self.chunks = tuple(self.metadata['chunk_grid']['configuration']['chunk_shape'])
        self.dtype = _dtype_from_metadata(self.metadata)
        if self.chunks != (1,) + self.shape[1:]:
            raise ValueError(f'Expected one full frame per chunk, found {self.chunks}')

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, t):
        t = int(t) % self.shape[0]
        path = self.path / 'c' / str(t) / '0' / '0' / '0'
        raw = decode_zarr_chunk(path.read_bytes(), self.metadata, math.prod(self.chunks)*self.dtype.itemsize)
        return np.frombuffer(raw, dtype=self.dtype).reshape(self.chunks)[0]

def find_competition_root():
    candidates = [Path('/kaggle/input/competitions') / COMPETITION_SLUG, Path('/kaggle/input') / COMPETITION_SLUG]
    for path in candidates:
        if (path / 'train').exists() and (path / 'test').exists():
            return path
    raise FileNotFoundError('Attach the official competition data.')

ROOT = find_competition_root()
TRAIN_DIR, TEST_DIR = ROOT / 'train', ROOT / 'test'
train_movies = sorted(TRAIN_DIR.glob('*.zarr'))
test_movies = sorted(TEST_DIR.glob('*.zarr'))
print(ROOT, '| train:', len(train_movies), '| test:', len(test_movies))


In [ ]:
def _first_existing(base, relative_paths):
    for rel in relative_paths:
        path = base / rel
        if (path / 'zarr.json').exists():
            return path
    raise FileNotFoundError(f'None of {relative_paths} found under {base}')

def _recursive_find_key(obj, target):
    if isinstance(obj, dict):
        if target in obj:
            return obj[target]
        for value in obj.values():
            found = _recursive_find_key(value, target)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = _recursive_find_key(value, target)
            if found is not None:
                return found
    return None

def load_geff(geff_path):
    geff_path = Path(geff_path)
    ids = read_zarr_v3_array(_first_existing(geff_path, ['nodes/ids'])).reshape(-1).astype(np.int64)
    props = {}
    for key in ['t','z','y','x']:
        props[key] = read_zarr_v3_array(_first_existing(geff_path, [f'nodes/props/{key}/values', f'nodes/{key}'])).reshape(-1)
    edge_path = _first_existing(geff_path, ['edges/ids'])
    edges = read_zarr_v3_array(edge_path).reshape(-1, 2).astype(np.int64)
    nodes = pd.DataFrame({'node_id': ids, **props})
    root_meta = json.loads((geff_path / 'zarr.json').read_text())
    estimated = _recursive_find_key(root_meta, 'estimated_number_of_nodes')
    return nodes, edges, float(estimated) if estimated is not None else np.nan


probe_nodes, probe_edges, probe_estimated = load_geff(train_movies[0].with_suffix('.geff'))
print(train_movies[0].stem, len(probe_nodes), len(probe_edges), probe_estimated)


## 3. High-recall proposals, centroid refinement and tissue motion

The detector removes slowly varying background and takes the maximum response across three physical LoG scales. A permissive threshold protects faint-cell sensitivity. Local weighted centroids reduce voxel-quantization error.

The organizer-provided total-node estimate is then divided across frames in proportion to proposal availability. This converts the permissive pool into a calibrated prediction rather than submitting every weak maximum. Coarse phase correlation estimates global tissue translation at low resolution.


In [ ]:
def sigma_vox(sigma_um):
    return tuple((float(sigma_um) / SPACING_UM).tolist())

def odd_window(radius_um):
    radius = np.maximum(1, np.ceil(float(radius_um) / SPACING_UM).astype(int))
    return tuple((2 * radius + 1).tolist())

def normalize_frame(frame):
    frame = np.asarray(frame, dtype=np.float32)
    sample = frame[::max(1, frame.shape[0]//32), ::4, ::4]
    lo, hi = np.quantile(sample, [0.01, 0.9995])
    return np.clip((frame-lo) / max(float(hi-lo), 1.0), 0, 1)

def proposal_response(image):
    background = gaussian_filter(image, sigma_vox(CONFIG['background_sigma_um']))
    foreground = np.clip(image-background, 0, None)
    responses = [-s*s*gaussian_laplace(foreground, sigma_vox(s)) for s in CONFIG['log_scales_um']]
    return np.maximum.reduce(responses).astype(np.float32, copy=False)

def refine_centers(coords, response):
    """Move an integer peak to the positive-response weighted local centroid."""
    radius = np.maximum(1, np.ceil(CONFIG['refine_radius_um']/SPACING_UM).astype(int))
    refined = np.empty((len(coords), 3), np.float32)
    for n, center in enumerate(coords):
        lo = np.maximum(0, center-radius)
        hi = np.minimum(response.shape, center+radius+1)
        patch = response[tuple(slice(a,b) for a,b in zip(lo,hi))]
        weight = np.clip(patch-np.median(patch), 0, None)
        if weight.sum() <= 1e-12:
            refined[n] = center
            continue
        grid = np.indices(patch.shape).reshape(3,-1).T + lo
        refined[n] = np.average(grid, axis=0, weights=weight.ravel())
    return refined

def detect_proposal_pool(frame):
    image = normalize_frame(frame)
    response = proposal_response(image)
    positive = response[response > 0]
    if not positive.size:
        return np.empty((0,3),np.float32), np.empty(0,np.float32)
    threshold = np.quantile(positive, CONFIG['proposal_quantile'])
    maxima = response == maximum_filter(response, size=odd_window(CONFIG['nms_um']), mode='nearest')
    coords = np.argwhere(maxima & (response >= threshold))
    scores = response[tuple(coords.T)] if len(coords) else np.empty(0,np.float32)
    if len(coords) > CONFIG['max_proposals_per_frame']:
        keep = np.argpartition(scores, -CONFIG['max_proposals_per_frame'])[-CONFIG['max_proposals_per_frame']:]
        coords, scores = coords[keep], scores[keep]
    order = np.argsort(scores)[::-1]
    coords, scores = coords[order], scores[order]
    return refine_centers(coords, response), scores.astype(np.float32)

def estimated_total_nodes(store_path):
    meta_path = Path(store_path) / 'zarr.json'
    if not meta_path.exists():
        return np.nan
    value = _recursive_find_key(json.loads(meta_path.read_text()), 'estimated_number_of_nodes')
    return float(value) if value is not None else np.nan

def allocate_frame_counts(pool_counts, target_total):
    """Allocate a movie-level node budget while respecting per-frame capacity."""
    capacity = np.asarray(pool_counts, dtype=int)
    if not np.isfinite(target_total):
        return capacity
    target = min(int(round(target_total)), int(capacity.sum()))
    if target <= 0 or capacity.sum() == 0:
        return np.zeros_like(capacity)
    ideal = target * capacity / capacity.sum()
    chosen = np.minimum(np.floor(ideal).astype(int), capacity)
    remainder = target-int(chosen.sum())
    priority = ideal-chosen
    while remainder > 0:
        available = np.flatnonzero(chosen < capacity)
        if not len(available): break
        take = available[np.argmax(priority[available])]
        chosen[take] += 1; priority[take] -= 1; remainder -= 1
    return chosen

def phase_shift_um(previous, current):
    """Estimate coarse whole-frame translation by phase correlation."""
    step = np.array([2, 8, 8])
    a = normalize_frame(previous)[::step[0],::step[1],::step[2]]
    b = normalize_frame(current)[::step[0],::step[1],::step[2]]
    a, b = a-a.mean(), b-b.mean()
    fa, fb = np.fft.rfftn(a), np.fft.rfftn(b)
    cross = fb*np.conj(fa); cross /= np.maximum(np.abs(cross), 1e-8)
    peak = np.array(np.unravel_index(np.argmax(np.fft.irfftn(cross, s=a.shape)), a.shape), dtype=float)
    peak[peak > np.array(a.shape)/2] -= np.array(a.shape)[peak > np.array(a.shape)/2]
    shift_um=peak*step*SPACING_UM
    return shift_um if np.linalg.norm(shift_um)<=6.0 else np.zeros(3,dtype=np.float32)

def prepare_movie_pools(path):
    movie = TimeChunkedZarr(path)
    pools, scores, drifts = [], [], [np.zeros(3, dtype=np.float32)]
    previous = None
    for t in range(len(movie)):
        frame = np.asarray(movie[t])
        coords, values = detect_proposal_pool(frame)
        pools.append(coords); scores.append(values)
        if previous is not None:
            drifts.append(phase_shift_um(previous, frame).astype(np.float32))
        previous = frame
    estimate = estimated_total_nodes(path)
    if not np.isfinite(estimate) and Path(path).with_suffix('.geff').exists():
        _, _, estimate = load_geff(Path(path).with_suffix('.geff'))
    if not np.isfinite(estimate):
        # Hidden-test GEFF metadata is unavailable. Transfer the nodes/frame
        # estimate from training movies of the same embryo prefix.
        rate = NODE_RATE_PRIOR.get(embryo_id(path), NODE_RATE_PRIOR['global'])
        estimate = rate*len(movie)
    return pools, scores, np.asarray(drifts), estimate

def select_movie(pools, scores, estimate, count_multiplier):
    counts = allocate_frame_counts([len(x) for x in pools], estimate*count_multiplier)
    return [(c[:n],s[:n]) for c,s,n in zip(pools,scores,counts)]


## 4. Learned motion prior and sparse assignment

True annotated edges provide displacement examples even though most cells are unlabelled. Their robust median and median absolute deviation define an anisotropic motion model in microns. Candidate successors are limited to the six most plausible neighbours, and Hungarian assignment finds the lowest-cost one-to-one solution for the complete frame pair.

Division edges are deliberately deferred in this version. They carry only one tenth of the final metric weight, and adding poorly calibrated forks can reduce precision.


In [ ]:
def embryo_id(path):
    return Path(path).stem.split('_')[0]

def balanced_movies(paths, per_embryo):
    groups = defaultdict(list)
    for path in paths: groups[embryo_id(path)].append(path)
    return [p for key in sorted(groups) for p in groups[key][:per_embryo]]

def learn_node_rate_prior(paths):
    """Transfer only coarse nodes/frame counts, never test-image labels."""
    rates=defaultdict(list)
    for path in paths:
        _,_,estimate=load_geff(path.with_suffix('.geff'))
        if np.isfinite(estimate): rates[embryo_id(path)].append(estimate/len(TimeChunkedZarr(path)))
    pooled=[value for values in rates.values() for value in values]
    prior={key:float(np.median(values)) for key,values in rates.items()}
    prior['global']=float(np.median(pooled))
    return prior

validation_movies=balanced_movies(train_movies,VALIDATION_MOVIES_PER_EMBRYO)
fit_movies=[p for p in train_movies if p not in validation_movies] or train_movies
NODE_RATE_PRIOR=learn_node_rate_prior(fit_movies)
print('Validation movies:',[p.stem for p in validation_movies])
print('Node-rate prior:',NODE_RATE_PRIOR)

def learn_motion_prior(paths):
    """Estimate a robust physical displacement scale from annotated true edges."""
    displacements = []
    for path in paths:
        nodes, edges, _ = load_geff(path.with_suffix('.geff'))
        xyz = nodes.set_index('node_id')[['z','y','x']]
        movie_displacements=[]
        for source, target in edges:
            if source in xyz.index and target in xyz.index:
                movie_displacements.append((xyz.loc[target].to_numpy()-xyz.loc[source].to_numpy())*SPACING_UM)
        if movie_displacements:
            movie_displacements=np.asarray(movie_displacements,dtype=np.float32)
            displacements.extend(movie_displacements-np.median(movie_displacements,axis=0))
    d = np.asarray(displacements, dtype=np.float32)
    center = np.zeros(3,dtype=np.float32)
    scale = 1.4826*np.median(np.abs(d-center), axis=0)
    scale = np.maximum(scale, np.array([0.8,0.5,0.5]))
    return {'center_um':center, 'scale_um':scale, 'n_edges':len(d)}

MOTION_PRIOR = learn_motion_prior(fit_movies)
print('Motion prior:', {k:(v.tolist() if hasattr(v,'tolist') else v) for k,v in MOTION_PRIOR.items()})

def candidate_costs(prev_coords, curr_coords, drift_um, prev_scores, curr_scores):
    prev_um, curr_um = prev_coords*SPACING_UM, curr_coords*SPACING_UM
    raw = np.linalg.norm(prev_um[:,None,:]-curr_um[None,:,:], axis=2)
    residual = curr_um[None,:,:]-(prev_um[:,None,:]+drift_um+MOTION_PRIOR['center_um'])
    standardized = np.sqrt(np.sum((residual/MOTION_PRIOR['scale_um'])**2, axis=2))
    residual_distance = np.linalg.norm(residual, axis=2)
    # Scores are converted to within-frame ranks because response magnitudes vary by frame.
    prev_rank = np.linspace(1,0,len(prev_scores),endpoint=False) if len(prev_scores) else np.empty(0)
    curr_rank = np.linspace(1,0,len(curr_scores),endpoint=False) if len(curr_scores) else np.empty(0)
    confidence = (prev_rank[:,None]+curr_rank[None,:])/2
    cost = standardized-CONFIG['score_weight']*confidence
    valid = (raw <= CONFIG['raw_gate_um']) & (residual_distance <= CONFIG['residual_gate_um'])
    # Keep only the k cheapest successors for each source node.
    if len(curr_coords) > CONFIG['candidate_k']:
        nearest = np.argpartition(np.where(valid,cost,np.inf), CONFIG['candidate_k']-1, axis=1)[:,:CONFIG['candidate_k']]
        sparse = np.zeros_like(valid)
        sparse[np.arange(len(prev_coords))[:,None], nearest] = True
        valid &= sparse
    return cost, valid, raw

def learned_motion_assignment(prev_coords, prev_ids, curr_coords, curr_ids,
                              drift_um, prev_scores, curr_scores):
    if not len(prev_coords) or not len(curr_coords): return []
    cost, valid, raw = candidate_costs(prev_coords,curr_coords,drift_um,prev_scores,curr_scores)
    rows, cols = linear_sum_assignment(np.where(valid,cost,1e6))
    return [(int(prev_ids[i]),int(curr_ids[j]),float(raw[i,j])) for i,j in zip(rows,cols) if valid[i,j]]

def distance_assignment(prev_coords,prev_ids,curr_coords,curr_ids):
    if not len(prev_coords) or not len(curr_coords): return []
    raw=np.linalg.norm(prev_coords[:,None,:]*SPACING_UM-curr_coords[None,:,:]*SPACING_UM,axis=2)
    valid=raw<=7.0
    rows,cols=linear_sum_assignment(np.where(valid,raw,1e6))
    return [(int(prev_ids[i]),int(curr_ids[j]),float(raw[i,j])) for i,j in zip(rows,cols) if valid[i,j]]

def process_prepared(selected, drifts, linker='learnedMotion'):
    nodes, edges, next_id = [], [], 1
    prev_coords=np.empty((0,3)); prev_scores=np.empty(0); prev_ids=np.empty(0,dtype=np.int64)
    for t,(coords,scores) in enumerate(selected):
        ids=np.arange(next_id,next_id+len(coords),dtype=np.int64); next_id += len(coords)
        nodes.extend((int(i),t,float(z),float(y),float(x),float(s)) for i,(z,y,x),s in zip(ids,coords,scores))
        if linker=='learnedMotion':
            links=learned_motion_assignment(prev_coords,prev_ids,coords,ids,drifts[t],prev_scores,scores)
        else:
            links=distance_assignment(prev_coords,prev_ids,coords,ids)
        edges.extend(links)
        prev_coords,prev_scores,prev_ids=coords,scores,ids
    return nodes,edges


## 5. Official edge-oriented validation

Complete held-out movies are separated by embryo. The local scorer follows the published edge matching and node-count adjustment; the official implementation remains authoritative.


In [ ]:
def score_edges(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_total):
    pred_df = pd.DataFrame(pred_nodes, columns=['node_id','t','z','y','x','score'])
    mapping = {}
    for t, gt_part in gt_nodes.groupby('t'):
        pred_part = pred_df[pred_df.t == t]
        if pred_part.empty or gt_part.empty: continue
        pred_xyz = pred_part[['z','y','x']].to_numpy(float)
        gt_xyz = gt_part[['z','y','x']].to_numpy(float)
        distances = np.linalg.norm((pred_xyz[:,None,:]-gt_xyz[None,:,:])*SPACING_UM, axis=2)
        rows, cols = linear_sum_assignment(distances)
        pred_ids = pred_part.node_id.to_numpy(); gt_ids = gt_part.node_id.to_numpy()
        for i,j in zip(rows,cols):
            if distances[i,j] <= MATCH_RADIUS_UM: mapping[int(pred_ids[i])] = int(gt_ids[j])

    gt_edge_set = {tuple(map(int,e)) for e in gt_edges}
    gt_out = {s for s,_ in gt_edge_set}; gt_in = {t for _,t in gt_edge_set}
    tp_pairs, fp = set(), 0
    for source,target,_ in pred_edges:
        ms, mt = mapping.get(source), mapping.get(target)
        pair = (ms,mt)
        if ms is not None and mt is not None and pair in gt_edge_set:
            tp_pairs.add(pair)
        elif (ms is not None and ms in gt_out) or (mt is not None and mt in gt_in):
            fp += 1
    tp = len(tp_pairs); fn = len(gt_edge_set - tp_pairs)
    jaccard = tp / max(tp+fp+fn, 1)
    ratio = (len(pred_nodes)-estimated_total)/estimated_total if np.isfinite(estimated_total) and estimated_total>0 else np.nan
    adjusted = max(0, jaccard*(1-0.1*ratio)) if np.isfinite(ratio) else jaccard
    return {'edge_tp':tp,'edge_fp':fp,'edge_fn':fn,'edge_jaccard':jaccard,'node_count':len(pred_nodes),
            'node_ratio_delta':ratio,'adjusted_edge_jaccard':adjusted,'matched_nodes':len(set(mapping.values()))}


In [ ]:
validation_rows = []
prepared_cache = {}

for movie_path in validation_movies:
    print('Preparing', movie_path.stem)
    # Detection responses are computed once; count variants only slice cached proposals.
    gt_nodes, gt_edges, estimated = load_geff(movie_path.with_suffix('.geff'))
    pools, scores, drifts, _ = prepare_movie_pools(movie_path)
    for multiplier in CONFIG['count_multipliers']:
        selected = select_movie(pools, scores, estimated, multiplier)
        for linker in ['distanceHungarian','learnedMotion']:
            started = time.time()
            nodes, edges = process_prepared(selected, drifts, linker)
            metrics = score_edges(nodes, edges, gt_nodes, gt_edges, estimated)
            validation_rows.append({'dataset':movie_path.stem,'count_multiplier':multiplier,'linker':linker,
                                    'seconds':time.time()-started,'pred_edges':len(edges),**metrics})

validation = pd.DataFrame(validation_rows)
validation.to_csv('/kaggle/working/dayThreeValidation.csv',index=False)
validation['weight']=validation.edge_tp+validation.edge_fp+validation.edge_fn
rows=[]
for (multiplier,linker),group in validation.groupby(['count_multiplier','linker']):
    tp,fp,fn=group[['edge_tp','edge_fp','edge_fn']].sum()
    adjusted=np.average(group.adjusted_edge_jaccard,weights=group.weight) if group.weight.sum() else np.nan
    rows.append({'count_multiplier':multiplier,'linker':linker,'adjusted_edge_jaccard':adjusted,
                 'edge_jaccard':tp/max(tp+fp+fn,1),'matched_nodes':group.matched_nodes.sum(),
                 'edge_tp':tp,'edge_fp':fp,'edge_fn':fn,'seconds':group.seconds.sum()})
validation_summary=pd.DataFrame(rows).sort_values('adjusted_edge_jaccard',ascending=False)
BEST_MULTIPLIER=float(validation_summary.iloc[0].count_multiplier)
BEST_LINKER=str(validation_summary.iloc[0].linker)
display(validation)
display(validation_summary)
print('Selected configuration:',BEST_MULTIPLIER,BEST_LINKER)


## 6. Visual sanity check

Cyan circles are predictions; yellow crosses are sparse annotations. Unmatched cyan predictions can still be real cells.


In [ ]:
viz_path=validation_movies[0]
viz_movie=TimeChunkedZarr(viz_path)
viz_gt,_,_=load_geff(viz_path.with_suffix('.geff'))
viz_t=int(viz_gt.t.median())
pools,scores,_,estimate=prepare_movie_pools(viz_path)
selected=select_movie(pools,scores,estimate,BEST_MULTIPLIER)
coords,_=selected[viz_t]
frame=np.asarray(viz_movie[viz_t]); truth=viz_gt.loc[viz_gt.t==viz_t,['z','y','x']].to_numpy()
sample=frame[::2,::4,::4]; vmin,vmax=np.quantile(sample,[.5,.999])
fig,ax=plt.subplots(figsize=(8,8)); ax.imshow(frame.max(axis=0),cmap='gray',vmin=vmin,vmax=vmax)
ax.scatter(coords[:,2],coords[:,1],s=10,facecolors='none',edgecolors='cyan',label='prediction')
ax.scatter(truth[:,2],truth[:,1],s=16,c='yellow',marker='+',label='sparse annotation')
ax.set_title(f'{viz_path.stem}, t={viz_t}: refined proposals'); ax.legend(); ax.axis('off'); plt.show()


## 7. Frozen test inference

After validation selects the count multiplier, each test movie is processed with the same detector, motion model and assignment rules.


In [ ]:
COLUMNS=['dataset','row_type','node_id','t','z','y','x','source_id','target_id']
all_parts,run_rows=[],[]
for movie_path in test_movies:
    print('Processing',movie_path.stem)
    started=time.time()
    pools,scores,drifts,estimate=prepare_movie_pools(movie_path)
    selected=select_movie(pools,scores,estimate,BEST_MULTIPLIER)
    nodes,edges=process_prepared(selected,drifts,BEST_LINKER)
    node_rows=[(movie_path.stem,'node',i,t,round(z),round(y),round(x),-1,-1) for i,t,z,y,x,s in nodes]
    edge_rows=[(movie_path.stem,'edge',-1,-1,-1,-1,-1,s,t) for s,t,d in edges]
    all_parts.append(pd.DataFrame(node_rows+edge_rows,columns=COLUMNS))
    run_rows.append({'dataset':movie_path.stem,'estimated_nodes':estimate,'nodes':len(nodes),
                     'edges':len(edges),'seconds':time.time()-started})

submission=pd.concat(all_parts,ignore_index=True)
submission.insert(0,'id',np.arange(len(submission),dtype=np.int64))
for col in ['id','node_id','t','z','y','x','source_id','target_id']:
    submission[col]=submission[col].astype(np.int64)
run_summary=pd.DataFrame(run_rows)
display(run_summary)


In [ ]:
def validate_submission(df,expected_datasets):
    assert list(df.columns)==['id']+COLUMNS
    assert not df.empty and df.id.is_unique
    assert set(df.dataset)==set(expected_datasets)
    for dataset,part in df.groupby('dataset'):
        nodes=part[part.row_type=='node']; edges=part[part.row_type=='edge']
        assert len(nodes) and nodes.node_id.is_unique
        times=dict(zip(nodes.node_id,nodes.t)); ids=set(times)
        assert set(edges.source_id)<=ids and set(edges.target_id)<=ids
        if len(edges):
            assert edges.source_id.value_counts().max()<=1
            assert edges.target_id.value_counts().max()<=1
            assert all(times[b]==times[a]+1 for a,b in zip(edges.source_id,edges.target_id))
    return True

assert validate_submission(submission,[p.stem for p in test_movies])
submission.to_csv('/kaggle/working/submission.csv',index=False)
run_summary.to_csv('/kaggle/working/runSummary.csv',index=False)
validation_summary.to_csv('/kaggle/working/dayThreeValidationSummary.csv',index=False)
with open('/kaggle/working/selectedConfiguration.json','w') as f:
    json.dump({'config':CONFIG,'countMultiplier':BEST_MULTIPLIER,'linker':BEST_LINKER,
               'motionCenterUm':MOTION_PRIOR['center_um'].tolist(),
               'motionScaleUm':MOTION_PRIOR['scale_um'].tolist()},f,indent=2)
print(f'Wrote {len(submission):,} rows to /kaggle/working/submission.csv')


## Outputs and interpretation

- `submission.csv`: file to submit;
- `runSummary.csv`: nodes, edges and runtime per test movie;
- `dayThreeValidation.csv`: held-out movie results for every node-count multiplier;
- `dayThreeValidationSummary.csv`: aggregated model-selection table;
- `selectedConfiguration.json`: the exact configuration used.

The validation table compares ordinary distance assignment with learned-motion assignment on exactly the same proposals. The broader comparison is Day 3 against Day 2 on adjusted edge Jaccard, matched annotated nodes, node-count deviation and runtime. A leaderboard increase alone does not identify which component worked.
